# **Data Preprocessing**

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

In [ ]:
baseDir = "D:/SeniorProject/UseableImages"

for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  i = 0
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    newDir = os.path.join(teamDir, f"{team}_{i}.jpg")
    try:
      if os.path.isfile(imageDir):
        print(f"File: {imageDir} already exists.")
      else:
        os.rename(imageDir, newDir)
    except Exception as e:
      print(e)
    i += 1

File: D:/UseableImages\76ers\76ers_0.jpg already exists.
File: D:/UseableImages\76ers\76ers_1.jpg already exists.
File: D:/UseableImages\76ers\76ers_10.jpg already exists.
File: D:/UseableImages\76ers\76ers_11.jpg already exists.
File: D:/UseableImages\76ers\76ers_12.jpg already exists.
File: D:/UseableImages\76ers\76ers_13.jpg already exists.
File: D:/UseableImages\76ers\76ers_14.jpg already exists.
File: D:/UseableImages\76ers\76ers_15.jpg already exists.
File: D:/UseableImages\76ers\76ers_16.jpg already exists.
File: D:/UseableImages\76ers\76ers_17.jpg already exists.
File: D:/UseableImages\76ers\76ers_18.jpg already exists.
File: D:/UseableImages\76ers\76ers_19.jpg already exists.
File: D:/UseableImages\76ers\76ers_2.jpg already exists.
File: D:/UseableImages\76ers\76ers_20.jpg already exists.
File: D:/UseableImages\76ers\76ers_21.jpg already exists.
File: D:/UseableImages\76ers\76ers_22.jpg already exists.
File: D:/UseableImages\76ers\76ers_23.jpg already exists.
File: D:/UseableI

# **YOLO Implementation**

In [2]:
from ultralytics import YOLO
import cv2
import os

In [3]:
model = YOLO('yolov8n.pt')

100%|██████████| 6.25M/6.25M [00:00<00:00, 24.5MB/s]


In [ ]:
baseDir = "D:/SeniorProject/UseableImages"
# Looping through every image to ensure there is no problem opening them after unzipping
for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    try:
      img = Image.open(imageDir)
    except:
      print(f"Problem with file: {imageDir} - removing this file from dataset")
      os.remove(imageDir)

Problem with file: D:/UseableImages\Celtics\Celtics_0.jpg - removing this file from dataset
Problem with file: D:/UseableImages\Hawks\Hawks_0.jpg - removing this file from dataset
Problem with file: D:/UseableImages\Pacers\Pacers_0.jpg - removing this file from dataset
Problem with file: D:/UseableImages\Pistons\Pistons_0.jpg - removing this file from dataset
Problem with file: D:/UseableImages\Raptors\Raptors_0.jpg - removing this file from dataset


In [ ]:
def cropImages(inputDir, outputDir):
  # Make the new directory if it doesn't exist
  if not os.path.exists("D:/SeniorProject/CroppedImages"):
    os.mkdir("D:/SeniorProject/CroppedImages")
  # Looping through every team directory to crop the images
  for root, _, files in os.walk(inputDir):

    currentPath = os.path.relpath(root, inputDir)
    outputPath = os.path.join(outputDir, currentPath)
    # Making team directory in new output directory if it doesn't exist
    if not os.path.exists(outputPath):
      os.makedirs(outputPath)
    # Implementing an object detector on every image
    for file in files:

      imgPath = os.path.join(root, file)
      img = cv2.imread(imgPath)

      results = model(img)

      boxes = results[0].boxes.xyxy.cpu().numpy()
      # If one person is detected, then crop to that person
      if len(boxes) == 1:
        x1, y1, x2, y2 = boxes[0]
        croppedImg = img[int(y1):int(y2), int(x1):int(x2)]
        cv2.imwrite(os.path.join(outputPath, file), croppedImg)
      # Otherwise, crop an image for the largest bounding box
      elif len(boxes) > 1:
        largeBox = max(boxes, key=lambda box: (box[2]-box[0])*(box[3]-box[1]))
        x1, y1, x2, y2 = map(int, largeBox)
        croppedImg = img[int(y1):int(y2), int(x1):int(x2)]
        cv2.imwrite(os.path.join(outputPath, file), croppedImg)


inputBase = "D:/SeniorProject/UseableImages"
outputBase = "D:/SeniorProject/CroppedImages"

# Executing the function above on all the images in the UseableImages directory
cropImages(inputBase, outputBase)


0: 384x640 6 persons, 1 sports ball, 72.7ms
Speed: 6.4ms preprocess, 72.7ms inference, 374.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 13.6ms
Speed: 1.4ms preprocess, 13.6ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 544x640 5 persons, 1 frisbee, 58.2ms
Speed: 2.2ms preprocess, 58.2ms inference, 2.3ms postprocess per image at shape (1, 3, 544, 640)

0: 640x448 1 person, 48.8ms
Speed: 2.0ms preprocess, 48.8ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 448)

0: 448x640 1 person, 54.8ms
Speed: 1.2ms preprocess, 54.8ms inference, 2.6ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 11 persons, 10.2ms
Speed: 1.4ms preprocess, 10.2ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 12 persons, 1 bottle, 7.0ms
Speed: 1.4ms preprocess, 7.0ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)

0: 640x640 2 persons, 1 sports ball, 14.6ms
Speed: 2.7ms preprocess, 14.6m

In [ ]:
def cropEveryPerson(inputDir, outputDir):
  # Make the new directory if it doesn't exist
  if not os.path.exists(outputDir):
    os.mkdir(outputDir)
  # Looping through every team directory to crop the images
  for root, _, files in os.walk(inputDir):

    currentPath = os.path.relpath(root, inputDir)
    outputPath = os.path.join(outputDir, currentPath)
    # Making team directory in new output directory if it doesn't exist
    if not os.path.exists(outputPath):
      os.makedirs(outputPath)
    # Implementing an object detector on every image
    for file in files:

      imgPath = os.path.join(root, file)
      img = cv2.imread(imgPath)

      results = model(img)

      boxes = results[0].boxes.xyxy.cpu().numpy()
      # If one person is detected, then crop to that person
      for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        croppedImg = img[int(y1):int(y2), int(x1):int(x2)]
        cv2.imwrite(os.path.join(outputPath, f"{file}_{i}.jpg"), croppedImg)

inputBase = "D:/SeniorProject/TempImages"
outputBase = "D:/SeniorProject/CroppedTempImages"

cropEveryPerson(inputBase, outputBase)

In [ ]:
baseDir = "D:/SeniorProject/CroppedImages"
# Looping through every team and image in the team's directory and resizing them
for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    try:
      img = Image.open(imageDir)
      image = img.resize((224, 224))
      image.save(imageDir)
    except:
      print(f"Problem with file: {imageDir} - removing this file from dataset")
      os.remove(imageDir)

In [ ]:
baseDir = "D:/SeniorProject/CroppedTempImages"
# Looping through every team and image in the team's directory and resizing them
for team in os.listdir(baseDir):
  i = 0
  teamDir = os.path.join(baseDir, team)
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    newImageDir = os.path.join(teamDir, f"{team}Temp_{i}.jpg")
    try:
      with Image.open(imageDir) as img:
        image = img.resize((224, 224))
        image.save(newImageDir)
      
      if imageDir != newImageDir:
        os.remove(imageDir)

      i += 1

    except Exception as e:
      print(f"Problem with file: {imageDir} - {e}")
      if os.path.exists(imageDir):
        os.remove(imageDir)

In [2]:
import requests
import base64
import json
import os

In [2]:
def encode_image(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

def makePrediction(imagePath):

    ollamaAPI = "http://localhost:11434/api/generate"
    
    payload = {
        "model": "llava",
        "prompt": "Is there an NBA Player wearing a jersey in this image? If so, what team are they on?",
        "images": [encode_image(imagePath)],
        "stream": True
    }

    response = requests.post(ollamaAPI, json=payload, stream=True)
    response_text = ""

    for chunk in response.iter_lines():
        if chunk:
            try:
                chunk_json = chunk.decode('utf-8')
                chunk_data = json.loads(chunk_json)
                response_text += chunk_data.get("response", "")

                if chunk_data.get("done"):
                    break
            except Exception as e:
                print(f"Error processing chunk: {e}")
                continue
            
    return response_text

In [3]:
teamDict = {
    "Philadelphia 76ers": "76ers", "76ers": "76ers", "Philadelphia": "76ers",
    "Milwaukee Bucks": "Bucks", "Bucks": "Bucks", "Milwaukee": "Bucks", 
    "Chicago Bulls": "Bulls", "Bulls": "Bulls", "Chicago": "Bulls",
    "Cleveland Cavaliers": "Cavaliers", "Cavaliers": "Cavaliers", "Cleveland": "Cavaliers",
    "Boston Celtics": "Celtics", "Celtics": "Celtics", "Boston": "Celtics",
    "Los Angeles Clippers": "Clippers", "Clippers": "Clippers",
    "Memphis Grizzlies": "Grizzlies", "Grizzlies": "Grizzlies", "Memphis": "Grizzlies", 
    "Atlanta Hawks": "Hawks", "Hawks": "Hawks", "Atlanta": "Hawks",
    "Miami Heat": "Heat", "Heat": "Heat", "Miami": "Heat",
    "Charlotte Hornets": "Hornets", "Hornets": "Hornets", "Charlotte": "Hornets",
    "Utah Jazz": "Jazz", "Jazz": "Jazz", "Utah": "Jazz",
    "Sacramento Kings": "Kings", "Kings": "Kings", "Sacramento": "Kings",
    "New York Knicks": "Knicks", "Knicks": "Knicks", "New York": "Knicks",
    "Los Angeles Lakers": "Lakers", "Lakers": "Lakers", "Los Angeles": "Lakers",
    "Orlando Magic": "Magic", "Magic": "Magic", "Orlando": "Magic",
    "Dallas Mavericks": "Mavericks", "Mavericks": "Mavericks", "Dallas": "Mavericks",
    "Brooklyn Nets": "Nets", "Nets": "Nets", "Brooklyn": "Nets",
    "Denver Nuggets": "Nuggets", "Nuggets": "Nuggets", "Denver": "Nuggets", 
    "Indiana Pacers": "Pacers", "Pacers": "Pacers", "Indiana": "Pacers",
    "New Orleans Pelicans": "Pelicans", "Pelicans": "Pelicans", "New Orleans": "Pelicans",
    "Detroit Pistons": "Pistons", "Pistons": "Pistons", "Detroit": "Pistons",
    "Toronto Raptors": "Raptors", "Raptors": "Raptors", "Toronto": "Raptors",
    "Houston Rockets": "Rockets", "Rockets": "Rockets", "Houston": "Rockets",
    "San Antonio Spurs": "Spurs", "Spurs": "Spurs", "San Antonio": "Spurs",
    "Phoenix Suns": "Suns", "Suns": "Suns", "Phoenix": "Suns",
    "Oklahoma City Thunder": "Thunder", "Thunder": "Thunder", "Oklahoma City": "Thunder",
    "Minnesota Timberwolves": "Timberwolves", "Timberwolves": "Timberwolves", "Minnesota": "Timberwolves",
    "Portland Trail Blazers": "Trail Blazers", "Trail Blazers": "Trail Blazers", "Portland": "Trail Blazers",
    "Golden State Warriors": "Warriors", "Warriors": "Warriors", "Golden State": "Warriors",
    "Washington Wizards": "Wizards", "Wizards": "Wizards", "Washington": "Wizards"
}

In [ ]:
teams = ["Wizards"]

for team in teams:
    for root, _, files in os.walk(f"D:/SeniorProject/CroppedTempImages/{team}"):
        for file in files:
            imgPath = os.path.join(root, file)
            prediction = makePrediction(imgPath)

            
            
            mappedPrediction = teamDict.get(prediction, "unknown")

            mappedPrediction = mappedPrediction.lower()

            for key in teamDict:
                if key.lower() in prediction.lower():
                    mappedPrediction = teamDict[key]

            if mappedPrediction != "unknown":
                outputDir = f"D:/SeniorProject/LLMImageLabels/{mappedPrediction}"
                if not os.path.exists(outputDir):
                    os.makedirs(outputDir)
                
                os.rename(imgPath, os.path.join(outputDir, file))
            else:
                if not os.path.exists("D:/SeniorProject/LLMImageLabels/Unkown"):
                    os.makedirs("D:/SeniorProject/LLMImageLabels/Unkown")
                os.rename(imgPath, "D:/SeniorProject/LLMImageLabels/Unkown/" + file)
